# 02 — Baselines

**Tham chiếu:** `docs/FinalTerm/pipeline_tong_the.md` Phase 3.

Cần có 3 baseline trước khi build fusion (để chứng minh fusion mang lại giá trị):

| # | Model | Modality | Bài báo nguồn |
|---|---|---|---|
| 1 | **ESC/EAS 2025 Rule** | Tabular (rule-based) | Mach et al. 2025 — guideline |
| 2 | **Tabular ML** (RF / XGBoost / MLP) | Tabular | Vu et al. 2025 |
| 3 | **CNN-only** (ResNet-50) | Image | Gao et al. 2026, He et al. 2024 |

**Metrics:** AUC-ROC · F1 · Sensitivity · Specificity · **Sensitivity trên nhóm discordant** (subgroup novelty).

**Strategy:** Stratified 5-fold CV cho RF/XGBoost/MLP/ESC Rule (chạy nhanh trên CPU). CNN-only chạy 1 fold 70/15/15 (prototype) hoặc 5-fold trên Colab GPU.

## 0. Setup — Local & Google Colab

In [ ]:
import os, sys
from pathlib import Path

try:
    import google.colab  # noqa: F401
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = Path('/content/drive/MyDrive/it2039-xulytinhieuhinhanhykhoa/project')
    assert PROJECT_ROOT.exists(), f'Không thấy project tại {PROJECT_ROOT}'
    !pip install -q xgboost pyyaml >/dev/null
else:
    PROJECT_ROOT = Path('..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('IN_COLAB     =', IN_COLAB)

In [ ]:
import warnings
warnings.filterwarnings('ignore', category=UserWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    roc_auc_score, f1_score, confusion_matrix, classification_report,
)
from sklearn.utils.class_weight import compute_class_weight

from src.clinical_rules import (
    annotate_dataframe, esceas_2025_rule_predict, LDL_C_GOAL_MG_DL,
)
from src.dataset import TABULAR_FEATURES, encode_sex
from src.utils import CSV_PATH, FIGURES_DIR, RESULTS_DIR, set_seed

sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 110
plt.rcParams['savefig.bbox'] = 'tight'

SEED = 42
N_SPLITS = 5
set_seed(SEED)

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
PREDICTIONS_DIR = RESULTS_DIR / 'predictions'
PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)

# Reuse annotated CSV từ 01_eda. Nếu chưa có, sinh lại
annotated_csv = RESULTS_DIR / 'carotid_annotated.csv'
if annotated_csv.exists():
    df = pd.read_csv(annotated_csv)
    print('Loaded annotated:', df.shape)
else:
    df = annotate_dataframe(pd.read_csv(CSV_PATH))
    df.to_csv(annotated_csv, index=False)
    print('Regenerated annotated:', df.shape)

assert int(df['is_discordant'].sum()) == 33, 'Discordant count check failed'
print(f'Discordant cases: {int(df.is_discordant.sum())}/300 ✓')

## Helper — bảng metrics chuẩn (gắn discordance subgroup)

Mọi baseline đều gọi hàm này để có cùng format kết quả.

In [ ]:
def metrics_full(y_true, y_prob, discordant_mask, threshold=0.5, name=''):
    """Trả về dict metrics: AUC / F1 / Sens / Spec + discordant subgroup."""
    y_pred = (y_prob >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    sens = tp / (tp + fn) if (tp + fn) else 0.0
    spec = tn / (tn + fp) if (tn + fp) else 0.0
    auc = roc_auc_score(y_true, y_prob) if len(set(y_true)) > 1 else float('nan')

    # Subgroup discordant
    dm = np.asarray(discordant_mask, dtype=bool)
    n_disc = int(dm.sum())
    if n_disc > 0:
        y_d, p_d = y_true[dm], y_prob[dm]
        yhat_d = (p_d >= threshold).astype(int)
        d_tn, d_fp, d_fn, d_tp = confusion_matrix(y_d, yhat_d, labels=[0, 1]).ravel()
        d_sens = d_tp / (d_tp + d_fn) if (d_tp + d_fn) else 0.0
        d_npv  = d_tn / (d_tn + d_fn) if (d_tn + d_fn) else 0.0
    else:
        d_sens = d_npv = float('nan')

    return {
        'model': name,
        'AUC': auc,
        'F1': f1_score(y_true, y_pred, zero_division=0),
        'Sens': sens,
        'Spec': spec,
        'Sens@discordant': d_sens,
        'NPV@discordant': d_npv,
        'n_discordant': n_disc,
    }

## Baseline 1 — ESC/EAS 2025 Rule (clinical, no DL)

Theo `clinical_rules.esceas_2025_rule_predict`:

```python
predict_positive = (LDL_C >= LDL_goal_of_category) OR (Lp(a) > 50 mg/dL)
```

**Lưu ý:** Rule này KHÔNG dùng `Plaque_present` để predict (vì plaque là target cần dự đoán). Đây là benchmark "không có deep learning" — cho thấy phương pháp screening truyền thống bắt được bao nhiêu, và miss bao nhiêu trong nhóm discordant.

In [ ]:
# Rule là deterministic — không có "probability", dùng prediction trực tiếp
yhat_rule = esceas_2025_rule_predict(df)
y_true = df['Plaque_present'].values
disc_mask = df['is_discordant'].values

# Coi prediction là "score" 0/1 để dùng chung hàm metrics
m_rule = metrics_full(y_true, yhat_rule.astype(float), disc_mask,
                      threshold=0.5, name='ESC/EAS Rule')
pd.Series(m_rule).to_frame('Value')

In [ ]:
# Phân tích lỗi: trong 33 discordant, rule miss bao nhiêu?
df_rule = df.copy()
df_rule['rule_pred'] = yhat_rule
df_rule['rule_correct'] = (df_rule['rule_pred'] == df_rule['Plaque_present']).astype(int)

disc_df = df_rule[df_rule['is_discordant']]
missed_disc = disc_df[(disc_df['Plaque_present'] == 1) & (disc_df['rule_pred'] == 0)]
print(f'Discordant cases với plaque=1: {(disc_df.Plaque_present == 1).sum()}')
print(f'  Rule predict đúng (TP): {((disc_df.Plaque_present==1) & (disc_df.rule_pred==1)).sum()}')
print(f'  Rule MISS (FN)        : {len(missed_disc)}')
print()
if len(missed_disc):
    print('Các case bị rule miss (Plaque có thật nhưng rule không bắt):')
    print(missed_disc[['Patient_ID', 'LDL_C_mg_dL', 'ldl_goal_mg_dl',
                       'Lp(a)_mg_dL', 'lpa_tier', 'discordance_subtype']].to_string(index=False))

In [ ]:
# Lưu predictions của ESC Rule
pred_rule = pd.DataFrame({
    'Patient_ID': df['Patient_ID'],
    'y_true': y_true,
    'y_prob': yhat_rule.astype(float),
    'y_pred': yhat_rule,
    'is_discordant': disc_mask,
    'fold': -1,  # rule chạy trên toàn dataset
})
pred_rule.to_csv(PREDICTIONS_DIR / 'pred_esceas_rule.csv', index=False)
print(f'Saved: {PREDICTIONS_DIR / "pred_esceas_rule.csv"}')

## Baseline 2 — Tabular ML (5-fold CV)

Train 3 model trên 9 features đã chốt ở `dataset.py::TABULAR_FEATURES`:
- **Random Forest** — robust, không cần scaling
- **XGBoost** — typically top performer trên tabular y khoa
- **MLP (sklearn)** — analog của MLPBranch fusion (nhỏ hơn, 64-32 hidden)

Stratified 5-fold CV, scaler fit trong từng fold tránh data leakage. Gộp OOF predictions để tính metric + subgroup analysis.

In [ ]:
# Chuẩn bị X / y
df_ml = df.copy()
df_ml['Sex'] = encode_sex(df_ml['Sex'])
X_all = df_ml[TABULAR_FEATURES].to_numpy(dtype=np.float32)
y_all = df_ml['Plaque_present'].to_numpy(dtype=np.int64)
disc_all = df_ml['is_discordant'].to_numpy(dtype=bool)

print(f'X shape: {X_all.shape}  (300 × 9 features)')
print(f'Features: {TABULAR_FEATURES}')
print(f'Class distribution: 0={np.sum(y_all==0)}, 1={np.sum(y_all==1)}')

# Class weights — sẽ dùng cho MLP
cls_weights = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_all)
print(f'Class weights (balanced): {cls_weights}')

In [ ]:
def run_tabular_cv(model_factory, name: str, needs_scaling: bool = False):
    """
    Stratified 5-fold CV trên tabular.
    Returns: (oof_df, fold_metrics_df)
    """
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    oof_prob = np.zeros(len(y_all), dtype=np.float32)
    oof_fold = -np.ones(len(y_all), dtype=np.int32)
    fold_metrics = []

    for k, (tr_idx, va_idx) in enumerate(skf.split(X_all, y_all), start=1):
        X_tr, X_va = X_all[tr_idx], X_all[va_idx]
        y_tr, y_va = y_all[tr_idx], y_all[va_idx]

        if needs_scaling:
            scaler = StandardScaler().fit(X_tr)
            X_tr, X_va = scaler.transform(X_tr), scaler.transform(X_va)

        clf = model_factory()
        clf.fit(X_tr, y_tr)
        p_va = clf.predict_proba(X_va)[:, 1]
        oof_prob[va_idx] = p_va
        oof_fold[va_idx] = k

        fm = metrics_full(y_va, p_va, disc_all[va_idx], name=f'{name} fold{k}')
        fm['fold'] = k
        fold_metrics.append(fm)
        print(f'  fold {k}: AUC={fm["AUC"]:.3f}  F1={fm["F1"]:.3f}  '
              f'Sens={fm["Sens"]:.3f}  Sens@disc={fm["Sens@discordant"]:.3f} '
              f'(n_disc={fm["n_discordant"]})')

    oof_df = pd.DataFrame({
        'Patient_ID': df['Patient_ID'].values,
        'y_true': y_all,
        'y_prob': oof_prob,
        'y_pred': (oof_prob >= 0.5).astype(int),
        'is_discordant': disc_all,
        'fold': oof_fold,
    })
    return oof_df, pd.DataFrame(fold_metrics)

In [ ]:
# 2a. Random Forest
print('Random Forest 5-fold:')
oof_rf, fm_rf = run_tabular_cv(
    lambda: RandomForestClassifier(n_estimators=300, max_depth=None,
                                    class_weight='balanced',
                                    random_state=SEED, n_jobs=-1),
    name='RF', needs_scaling=False)
oof_rf.to_csv(PREDICTIONS_DIR / 'pred_rf.csv', index=False)

In [ ]:
# 2b. XGBoost
from xgboost import XGBClassifier

scale_pos_weight = (y_all == 0).sum() / max((y_all == 1).sum(), 1)
print(f'XGBoost 5-fold (scale_pos_weight={scale_pos_weight:.2f}):')
oof_xgb, fm_xgb = run_tabular_cv(
    lambda: XGBClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=scale_pos_weight,
        eval_metric='auc', random_state=SEED, n_jobs=-1, verbosity=0),
    name='XGB', needs_scaling=False)
oof_xgb.to_csv(PREDICTIONS_DIR / 'pred_xgb.csv', index=False)

In [ ]:
# 2c. MLP (sklearn) — analog của MLPBranch
print('MLP (sklearn) 5-fold:')
oof_mlp, fm_mlp = run_tabular_cv(
    lambda: MLPClassifier(
        hidden_layer_sizes=(64, 128, 64),
        activation='relu', solver='adam',
        alpha=1e-4,  # L2
        batch_size=16, learning_rate_init=1e-3,
        max_iter=300, early_stopping=True, validation_fraction=0.15,
        random_state=SEED),
    name='MLP', needs_scaling=True)
oof_mlp.to_csv(PREDICTIONS_DIR / 'pred_mlp.csv', index=False)

In [ ]:
# Tổng hợp metrics cho 3 model — mean ± std AUC/F1/Sens/Spec
def summarize_folds(fm_df: pd.DataFrame, name: str) -> dict:
    cols = ['AUC', 'F1', 'Sens', 'Spec', 'Sens@discordant', 'NPV@discordant']
    out = {'model': name}
    for c in cols:
        m, s = fm_df[c].mean(), fm_df[c].std()
        out[c] = f'{m:.3f} ± {s:.3f}'
    out['n_disc_total'] = int(fm_df['n_discordant'].sum())
    return out

tabular_summary = pd.DataFrame([
    summarize_folds(fm_rf, 'RF'),
    summarize_folds(fm_xgb, 'XGBoost'),
    summarize_folds(fm_mlp, 'MLP'),
])
tabular_summary

In [ ]:
# Feature importance — RandomForest và XGBoost (fit lần nữa trên toàn dataset)
rf_full = RandomForestClassifier(n_estimators=300, class_weight='balanced',
                                  random_state=SEED, n_jobs=-1).fit(X_all, y_all)
xgb_full = XGBClassifier(n_estimators=300, max_depth=4, learning_rate=0.05,
                          scale_pos_weight=scale_pos_weight, eval_metric='auc',
                          random_state=SEED, n_jobs=-1, verbosity=0).fit(X_all, y_all)

imp_df = pd.DataFrame({
    'feature': TABULAR_FEATURES,
    'RF_importance': rf_full.feature_importances_,
    'XGB_importance': xgb_full.feature_importances_,
}).sort_values('RF_importance', ascending=False)

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(imp_df))
w = 0.4
ax.barh(x - w/2, imp_df['RF_importance'], w, label='RF', color='#4c72b0')
ax.barh(x + w/2, imp_df['XGB_importance'], w, label='XGBoost', color='#dd8452')
ax.set_yticks(x); ax.set_yticklabels(imp_df['feature'])
ax.invert_yaxis(); ax.legend()
ax.set_xlabel('Feature importance')
ax.set_title('Tabular feature importance — RF vs XGBoost')
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_feature_importance.png')
plt.show()

imp_df.round(3)

## Baseline 3 — CNN-only (ResNet-50)

Chỉ dùng nhánh ảnh, không có tabular data. So sánh trực tiếp với CNN branch của fusion model.

**Kiến trúc:** ResNet-50 (ImageNet pretrained) → GAP 2048 → Linear(2048, 2). Train với weighted CE để xử lý imbalance 2.16:1.

**Aggregation:** Bệnh nhân case có 5 ảnh → mean-pool embedding như fusion model.

**Fine-tuning 3 giai đoạn** (đồng nhất với pipeline Phase 5):
1. Freeze backbone, train head (5 epochs)
2. Unfreeze layer4 (15 epochs)
3. Unfreeze all, lr nhỏ (10 epochs)

**Yêu cầu:** GPU. Trên CPU sẽ rất chậm (>2 giờ cho 1 fold) — skip cell training nếu không có CUDA, chỉ giữ inference.

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.dataset import CarotidDataset, build_transforms, carotid_collate
from src.models.cnn_branch import CNNBranch
from src.utils import get_device, CHECKPOINTS_DIR

CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)
device = get_device()
print('Device:', device)
if device.type == 'cpu':
    print('⚠️  Không có CUDA — CNN-only sẽ rất chậm. Nên chạy trên Colab GPU.')
    print('   Skip training, chạy lại notebook trên Colab để fill phần này.')

In [ ]:
class CNNOnlyModel(nn.Module):
    """Baseline CNN: ResNet-50 + projection 128 + head 2-class.
       Giữ projection để fair comparison với CNN branch của fusion."""
    def __init__(self, pretrained: bool = True):
        super().__init__()
        self.cnn = CNNBranch(embed_dim=128, pretrained=pretrained, dropout=0.3)
        self.head = nn.Sequential(
            nn.Linear(128, 64),
            nn.BatchNorm1d(64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.4),
            nn.Linear(64, 2),
        )

    def forward(self, images, n_images):
        emb = self.cnn(images, n_images)
        return self.head(emb)

In [ ]:
def train_cnn_only(df_train, df_val, *, cfg, device, fold_label='proto'):
    """Train CNN-only baseline với 3-stage finetune. Trả về (model, val_probs, val_true, val_disc)."""
    set_seed(cfg['seed'])

    ds_tr = CarotidDataset(df_train, image_transform=build_transforms(train=True))
    ds_va = CarotidDataset(df_val,   image_transform=build_transforms(train=False))

    loader_tr = DataLoader(ds_tr, batch_size=cfg['batch_size'], shuffle=True,
                            collate_fn=carotid_collate, num_workers=0, pin_memory=True)
    loader_va = DataLoader(ds_va, batch_size=cfg['batch_size'], shuffle=False,
                            collate_fn=carotid_collate, num_workers=0, pin_memory=True)

    model = CNNOnlyModel(pretrained=True).to(device)

    # Weighted CE
    y_tr = df_train['Plaque_present'].values
    w = compute_class_weight('balanced', classes=np.array([0, 1]), y=y_tr)
    criterion = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32, device=device))

    def make_opt(lr_backbone, lr_new):
        return torch.optim.AdamW([
            {'params': model.cnn.backbone.parameters(),  'lr': lr_backbone},
            {'params': model.cnn.projection.parameters(),'lr': lr_new},
            {'params': model.head.parameters(),          'lr': lr_new},
        ], weight_decay=cfg['weight_decay'])

    history = []
    best_auc, best_state = -1.0, None

    def run_epoch(loader, train: bool, opt=None):
        model.train(train)
        losses, ys, ps = [], [], []
        for batch in loader:
            imgs = batch['images'].to(device, non_blocking=True)
            n_imgs = batch['n_images'].to(device)
            y = batch['plaque'].to(device)
            with torch.set_grad_enabled(train):
                logits = model(imgs, n_imgs)
                loss = criterion(logits, y)
                if train:
                    opt.zero_grad(); loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), cfg['gradient_clip'])
                    opt.step()
            losses.append(loss.item())
            ys.append(y.detach().cpu().numpy())
            ps.append(torch.softmax(logits, dim=1)[:, 1].detach().cpu().numpy())
        return float(np.mean(losses)), np.concatenate(ys), np.concatenate(ps)

    # ---- Stage 1: freeze backbone ----
    print(f'[{fold_label}] Stage 1 — freeze backbone, {cfg["stage1_epochs"]} epochs')
    model.cnn.freeze_backbone()
    opt = make_opt(cfg['lr_backbone'], cfg['lr_new_layers'])
    for ep in range(1, cfg['stage1_epochs'] + 1):
        tr_loss, _, _ = run_epoch(loader_tr, train=True, opt=opt)
        va_loss, va_y, va_p = run_epoch(loader_va, train=False)
        auc = roc_auc_score(va_y, va_p) if len(set(va_y)) > 1 else float('nan')
        history.append({'stage': 1, 'epoch': ep, 'tr_loss': tr_loss, 'va_loss': va_loss, 'va_auc': auc})
        print(f'  S1 ep{ep:02d}: tr_loss={tr_loss:.4f}  va_loss={va_loss:.4f}  va_auc={auc:.3f}')
        if auc > best_auc:
            best_auc = auc; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    # ---- Stage 2: unfreeze layer4 ----
    print(f'[{fold_label}] Stage 2 — unfreeze layer4, {cfg["stage2_epochs"]} epochs')
    model.cnn.unfreeze_layer4()
    opt = make_opt(cfg['lr_backbone'], cfg['lr_new_layers'])
    for ep in range(1, cfg['stage2_epochs'] + 1):
        tr_loss, _, _ = run_epoch(loader_tr, train=True, opt=opt)
        va_loss, va_y, va_p = run_epoch(loader_va, train=False)
        auc = roc_auc_score(va_y, va_p) if len(set(va_y)) > 1 else float('nan')
        history.append({'stage': 2, 'epoch': ep, 'tr_loss': tr_loss, 'va_loss': va_loss, 'va_auc': auc})
        print(f'  S2 ep{ep:02d}: tr_loss={tr_loss:.4f}  va_loss={va_loss:.4f}  va_auc={auc:.3f}')
        if auc > best_auc:
            best_auc = auc; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    # ---- Stage 3: unfreeze all, smaller lr ----
    print(f'[{fold_label}] Stage 3 — unfreeze all, lr/10, {cfg["stage3_epochs"]} epochs')
    model.cnn.unfreeze_all()
    opt = make_opt(cfg['lr_backbone'] / 10, cfg['lr_new_layers'] / 10)
    for ep in range(1, cfg['stage3_epochs'] + 1):
        tr_loss, _, _ = run_epoch(loader_tr, train=True, opt=opt)
        va_loss, va_y, va_p = run_epoch(loader_va, train=False)
        auc = roc_auc_score(va_y, va_p) if len(set(va_y)) > 1 else float('nan')
        history.append({'stage': 3, 'epoch': ep, 'tr_loss': tr_loss, 'va_loss': va_loss, 'va_auc': auc})
        print(f'  S3 ep{ep:02d}: tr_loss={tr_loss:.4f}  va_loss={va_loss:.4f}  va_auc={auc:.3f}')
        if auc > best_auc:
            best_auc = auc; best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    # Load best checkpoint
    if best_state is not None:
        model.load_state_dict(best_state)

    # Final val predictions
    _, va_y, va_p = run_epoch(loader_va, train=False)
    disc_va = df_val['is_discordant'].values

    return model, va_p, va_y, disc_va, pd.DataFrame(history)


CNN_CFG = dict(
    seed=SEED, batch_size=16, weight_decay=1e-4, gradient_clip=1.0,
    lr_backbone=1e-5, lr_new_layers=1e-4,
    stage1_epochs=5, stage2_epochs=15, stage3_epochs=10,
)

### 3a. Prototype run — 70/15/15 split (kiểm tra nhanh)

Chạy 1 lần trên split holdout để verify training loop work end-to-end. Kết quả "chấp nhận được" cho prototype, dùng để debug. **Báo cáo cuối kỳ nên dùng 5-fold CV** (cell tiếp theo).

In [ ]:
from src.dataset import stratified_split

if device.type == 'cuda':
    df_tr, df_va, df_te = stratified_split(df, test_size=0.15, val_size=0.15, seed=SEED)
    print(f'Train/Val/Test: {len(df_tr)}/{len(df_va)}/{len(df_te)}')

    model_cnn, va_p, va_y, va_disc, hist = train_cnn_only(df_tr, df_va, cfg=CNN_CFG, device=device, fold_label='proto')

    m_proto = metrics_full(va_y, va_p, va_disc, name='CNN-only (proto)')
    pd.Series(m_proto).to_frame('Value').T
else:
    print('Skip CNN prototype training — no GPU. Chạy trên Colab.')

### 3b. Full 5-fold CV (báo cáo cuối)

5 fold × 30 epoch ≈ vài giờ trên Colab T4. Lưu predictions OOF để 03_fusion.ipynb so sánh.

In [ ]:
if device.type == 'cuda':
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)
    oof_prob = np.zeros(len(df), dtype=np.float32)
    oof_fold = -np.ones(len(df), dtype=np.int32)
    fm_cnn = []

    for k, (tr_idx, va_idx) in enumerate(skf.split(np.zeros(len(df)), df['Plaque_present'].values), start=1):
        print(f'\n========== FOLD {k}/{N_SPLITS} ==========')
        df_tr_k = df.iloc[tr_idx].reset_index(drop=True)
        df_va_k = df.iloc[va_idx].reset_index(drop=True)

        model_k, va_p, va_y, va_disc, hist_k = train_cnn_only(
            df_tr_k, df_va_k, cfg=CNN_CFG, device=device, fold_label=f'fold{k}')

        oof_prob[va_idx] = va_p
        oof_fold[va_idx] = k
        fm = metrics_full(va_y, va_p, va_disc, name=f'CNN-only fold{k}')
        fm['fold'] = k
        fm_cnn.append(fm)
        print(f'  → fold {k}: AUC={fm["AUC"]:.3f}  F1={fm["F1"]:.3f}  '
              f'Sens={fm["Sens"]:.3f}  Sens@disc={fm["Sens@discordant"]:.3f}')

        # Save checkpoint per fold
        torch.save(model_k.state_dict(), CHECKPOINTS_DIR / f'cnn_only_fold{k}.pt')

    fm_cnn = pd.DataFrame(fm_cnn)
    cnn_summary = summarize_folds(fm_cnn, 'CNN-only')
    print('\n========== CNN-only 5-fold summary ==========')
    print(pd.Series(cnn_summary).to_string())

    # Save OOF predictions
    oof_cnn = pd.DataFrame({
        'Patient_ID': df['Patient_ID'].values,
        'y_true': df['Plaque_present'].values,
        'y_prob': oof_prob,
        'y_pred': (oof_prob >= 0.5).astype(int),
        'is_discordant': df['is_discordant'].values,
        'fold': oof_fold,
    })
    oof_cnn.to_csv(PREDICTIONS_DIR / 'pred_cnn_only.csv', index=False)
    print(f'\nSaved: {PREDICTIONS_DIR / "pred_cnn_only.csv"}')
else:
    print('Skip CNN 5-fold — no GPU. Chạy trên Colab.')
    cnn_summary = None

## Tổng hợp 3 baseline + Discordance subgroup analysis

In [ ]:
# Bảng so sánh chính
rows = [
    {'model': 'ESC/EAS Rule', 'AUC': f"{m_rule['AUC']:.3f}", 'F1': f"{m_rule['F1']:.3f}",
     'Sens': f"{m_rule['Sens']:.3f}", 'Spec': f"{m_rule['Spec']:.3f}",
     'Sens@discordant': f"{m_rule['Sens@discordant']:.3f}",
     'NPV@discordant':  f"{m_rule['NPV@discordant']:.3f}",
     'n_disc_total': m_rule['n_discordant']},
    summarize_folds(fm_rf, 'RF (5-fold)'),
    summarize_folds(fm_xgb, 'XGBoost (5-fold)'),
    summarize_folds(fm_mlp, 'MLP (5-fold)'),
]
if cnn_summary is not None:
    rows.append(cnn_summary)

baseline_table = pd.DataFrame(rows)
baseline_table.to_csv(RESULTS_DIR / 'baseline_summary.csv', index=False)
baseline_table

In [ ]:
# Bar chart so sánh AUC + Sens@discordant
def parse_mean(s):
    """Parse 'm ± s' -> float m. Nếu là số đơn (ESC Rule) trả về float."""
    if isinstance(s, str) and '±' in s:
        return float(s.split('±')[0].strip())
    return float(s)

df_plot = baseline_table.copy()
df_plot['AUC_num'] = df_plot['AUC'].apply(parse_mean)
df_plot['Sens_disc_num'] = df_plot['Sens@discordant'].apply(parse_mean)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
colors = ['#888', '#4c72b0', '#2e6094', '#7a9cc4', '#dd8452'][:len(df_plot)]

ax = axes[0]
ax.bar(df_plot['model'], df_plot['AUC_num'], color=colors)
for i, v in enumerate(df_plot['AUC_num']):
    ax.text(i, v + 0.01, f'{v:.3f}', ha='center')
ax.set_ylim(0, 1.05)
ax.set_ylabel('AUC-ROC')
ax.set_title('Baseline AUC comparison')
ax.tick_params(axis='x', rotation=15)

ax = axes[1]
ax.bar(df_plot['model'], df_plot['Sens_disc_num'], color=colors)
for i, v in enumerate(df_plot['Sens_disc_num']):
    ax.text(i, v + 0.01, f'{v:.3f}', ha='center')
ax.set_ylim(0, 1.05)
ax.set_ylabel('Sensitivity on discordant subgroup')
ax.set_title('Sens@discordant (NOVELTY metric)')
ax.tick_params(axis='x', rotation=15)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig_baseline_comparison.png')
plt.show()

## Tổng kết Phase 3

**Saved predictions** (sẽ load lại ở `03_fusion.ipynb` để so sánh với Fusion):
- `results/predictions/pred_esceas_rule.csv` — ESC/EAS Rule (full dataset, 0/1 prediction)
- `results/predictions/pred_rf.csv` — Random Forest OOF (5-fold)
- `results/predictions/pred_xgb.csv` — XGBoost OOF (5-fold)
- `results/predictions/pred_mlp.csv` — MLP-sklearn OOF (5-fold)
- `results/predictions/pred_cnn_only.csv` — CNN-only OOF (5-fold, **chỉ khi chạy trên GPU**)

**Saved figures:**
- `fig_feature_importance.png` — RF vs XGBoost tabular importance
- `fig_baseline_comparison.png` — AUC + Sens@discordant comparison

**Saved tables:**
- `results/baseline_summary.csv` — bảng tổng hợp metrics 3 baseline
- Checkpoints CNN-only: `results/checkpoints/cnn_only_fold{k}.pt`

→ Sẵn sàng chuyển sang **Phase 4–6** (`03_fusion.ipynb`): train Multimodal Fusion, ablation study, và **discordance subgroup analysis** đối chiếu với 3 baseline.